# Backtest EMACross

Run the EMACross strategy on a simulated exchange venue.

**Sections:**
1. Setup — imports, catalog, engine configuration
2. Single run — fills, positions, account reports
3. Visualisation — price + EMA overlay with trade markers, equity curve
4. Statistics — analyzer performance summary
5. Parameter sweep — fast/slow grid with heatmap

**Prerequisites:** Run `scripts/fetch_binance_candles.py` first to populate `data/catalog/`.

In [ ]:
# ── Cell 1: Imports + shared config ────────────────────────────────
#
# All tuneable values live here. Change once, used everywhere below.

from decimal import Decimal

import pandas as pd
from IPython.display import HTML

# create_tearsheet removed — see Cell 7 for why
from nautilus_trader.model.data import BarType
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.backtesting import make_engine, run_sweep
from src.backtesting.analysis import (
    performance_by_year, performance_by_regime, tag_regimes,
    run_alt_instrument_check, run_fee_sweep,
)
from src.backtesting.baselines import (
    buy_and_hold, random_entry_baseline,
)
from src.backtesting.engine import resolve_strategy_liquidation_config
from src.core import (
    TOPIC_ACCOUNT_LIQUIDATED,
    TOPIC_POSITION_LIQUIDATED,
    LiquidationConfig,
    SizingConfig,
    bar_type_str,
    get_venue_config,
    with_venue_config,
)
from src.strategies.ma_cross import MACross, MACrossConfig

import sys
sys.path.insert(0, str(__import__("pathlib").Path("..").resolve()))

from charts import (
    plot_ma_cross, plot_equity_curve, print_summary_stats,
    plot_pnl_heatmap, generate_backtest_html,
    # v2 additions
    plot_trade_distributions,
    plot_yearly_breakdown, print_yearly_breakdown,
    plot_baselines_comparison,
    generate_sweep_html,
    generate_v2_tearsheet,
)
from utils import (
    make_instrument_id, save_notebook, save_notebook_html,
    # backtest setup
    load_backtest_data, print_setup_summary,
    print_liquidation_resolution,
    # run-time diagnostics
    print_run_diagnostics,
    # baselines orchestrator
    baselines_for_strategy, print_baselines_verdict,
    # sweep diagnostics
    print_sweep_liquidation_diagnostics,
)

# ── Shared config ────────────────────────────────────────────────
DATA_SOURCE      = "BINANCE_PERP"       # where the catalog data comes from
EXEC_VENUE       = "HYPERLIQUID_PERP"   # exchange we're simulating execution on
ASSET            = "BTC"
BAR_INTERVAL     = "1d"
DATE_START       = None                 # e.g. "2025-01-01" — None means "from start of data"
DATE_END         = None                 # e.g. "2025-04-01" — None means "to end of data"
MA_TYPE          = "EMA"                # strategy variant — do not change
STARTING_CAPITAL = 1000
TRADE_SIZE       = 2000                  # $100 USD notional per trade (back-compat path)
LEVERAGE         = 20
SAVE_TEARSHEET   = False

# MA params
FAST_MA  = 10
SLOW_MA  = 40

# Sweep grids
FAST_PERIODS = sorted(set([5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100] + [FAST_MA]))
SLOW_PERIODS = sorted(set([10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100, 200] + [SLOW_MA]))

# ── Sizing (Project A) ────────────────────────────────────────—
# None → fall back to TRADE_SIZE/trade_notional via SizingConfig(mode="fixed").
# To exercise equity-fraction sizing instead:
#   SIZING = SizingConfig(
#       mode="equity_frac",
#       risk_frac=Decimal("0.10"),    # 10% of equity at risk per trade
#       stop_pct=Decimal("0.05"),     # 5% protective stop
#       min_notional=Decimal("50"),
#   )
SIZING: SizingConfig | None = None

# ── Liquidation simulator (Project B) ────────────────────────—
# enabled=True wires up:
#   - LiquidationAware mixin (places a reduce-only stop at the cross-margin
#     liq price for every position), AND
#   - AccountAliveMonitor actor (halts the RiskEngine when equity falls
#     below the floor required to open a min-size order).
#
# With STARTING_CAPITAL=1000 and TRADE_SIZE=100, gross leverage is 0.1× —
# this config will NOT trigger any liquidations. To exercise the full
# path, raise TRADE_SIZE (e.g. 2000 → 2× gross leverage → liq at ~49.5%
# adverse) or switch SIZING above to equity_frac mode.
#
# Set enabled=False to run the legacy path with no liquidation simulation
# (back-compat with old sweep parquets).
LIQUIDATION = LiquidationConfig(
    enabled=True,
    halt_on_account_liquidation=True,
    # mm_rate, fee_rate, min_trade_notional all None — resolved at engine
    # build time from VENUE_CFG (mm_rate, taker_fee) and SIZING/instrument.
)

# Derived values
DATA_CFG       = get_venue_config(DATA_SOURCE) # Resolve venue configs —
VENUE_CFG      = get_venue_config(EXEC_VENUE)  # fees and leverage come from the registry
TRADE_NOTIONAL = Decimal(TRADE_SIZE)
INSTRUMENT_ID  = make_instrument_id(ASSET, DATA_SOURCE)
TRADING_PAIR   = INSTRUMENT_ID.split("-")[0]
BAR_TYPE_STR   = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)
VENUE          = Venue(DATA_CFG.nt_venue)

# ── File naming convention ─────────────────────────────────────
# SWEEP_NAME  = strategy + instrument + interval (NO params)
#               → deterministic; overwrites on re-run.
#               Used by:
#                 • data/sweeps/{SWEEP_NAME}.parquet              (run_sweep)
#                 • reports/sweeps/{SWEEP_NAME}_sweep.html        (sortable HTML)
# RESULT_NAME = SWEEP_NAME + "_f{FAST_MA}_s{SLOW_MA}" (per-config)
#               → per-run snapshot; appends _{timestamp}.html.
#               Used by:
#                 • reports/charts/{RESULT_NAME}_chart_{ts}.html      (TVLC chart)
#                 • reports/tearsheets/{RESULT_NAME}_tearsheet_{ts}.html  (v2 tearsheet)
CATALOG_PATH = "../../data/catalog"
SWEEP_NAME   = f"MACross-{MA_TYPE}_{ASSET}_{EXEC_VENUE}_{BAR_INTERVAL}"
RESULT_NAME  = f"{SWEEP_NAME}_f{FAST_MA}_s{SLOW_MA}"

print("Imports OK")
print(f"Sizing       : {'equity_frac' if SIZING and SIZING.mode == 'equity_frac' else 'fixed (trade_notional back-compat)'}")
print(f"Liquidation  : {'ENABLED' if LIQUIDATION.enabled else 'disabled'}")

In [ ]:
# ── Cell 2: Load data + resolve liquidation config ────────────────

instrument, bars = load_backtest_data(
    catalog_path=CATALOG_PATH,
    instrument_id=INSTRUMENT_ID,
    bar_type_str=BAR_TYPE_STR,
    venue_config=VENUE_CFG,           # overrides instrument fees with exec-venue rates
    date_start=DATE_START,
    date_end=DATE_END,
)
CURRENCY = instrument.settlement_currency

# Resolve LIQUIDATION fields (mm_rate, fee_rate, min_trade_notional) from
# VENUE_CFG / SIZING / instrument.  When SIZING is None we synthesize a
# fixed-mode SizingConfig from TRADE_NOTIONAL purely so the resolver has
# a min_trade_notional fallback.
LIQ_RESOLVED = resolve_strategy_liquidation_config(
    user=LIQUIDATION,
    venue_config=VENUE_CFG,
    instrument=instrument,
    sizing=SIZING or SizingConfig(mode="fixed", fixed_notional=TRADE_NOTIONAL),
)

print_setup_summary(
    instrument, bars,
    data_source=DATA_SOURCE, exec_venue=EXEC_VENUE, leverage=LEVERAGE,
)
print_liquidation_resolution(LIQ_RESOLVED, leverage=LEVERAGE)

In [ ]:
# ── Cell 3: Configure engine + venue ──────────────────────────────
engine = make_engine(
    VENUE, instrument, bars, STARTING_CAPITAL,
    leverage=LEVERAGE,
    liquidation=LIQ_RESOLVED,
    venue_config=VENUE_CFG,
    sizing=SIZING,
)
print("Engine configured.")

# Confirm AccountAliveMonitor wiring when liquidation is enabled.
if LIQ_RESOLVED is not None and LIQ_RESOLVED.enabled:
    actor_ids = [str(a) for a in engine.kernel.trader.actor_ids()]
    aam = [a for a in actor_ids if "AccountAlive" in a]
    print(f"  Registered actors  : {actor_ids}")
    print(f"  AccountAliveMonitor: {'✓ present' if aam else '✗ MISSING'}")
    print(f"  RiskEngine state   : {engine.kernel.risk_engine.trading_state}")

In [ ]:
# ── Cell 3b: Subscribe to liquidation events for diagnostics ─────
#
# The mixin publishes on TOPIC_POSITION_LIQUIDATED whenever a liquidation
# stop fills; AccountAliveMonitor publishes on TOPIC_ACCOUNT_LIQUIDATED
# when the alive predicate flips false.  Capture both for inspection
# after engine.run().

position_liquidations: list = []
account_liquidations: list = []

engine.kernel.msgbus.subscribe(
    topic=TOPIC_POSITION_LIQUIDATED,
    handler=position_liquidations.append,
)
engine.kernel.msgbus.subscribe(
    topic=TOPIC_ACCOUNT_LIQUIDATED,
    handler=account_liquidations.append,
)
print("Subscribed to liquidation topics — events will accumulate during run.")

In [ ]:
# ── Cell 4: Add MACross strategy + run ────────────────────────────
config = MACrossConfig(
    instrument_id=instrument.id,
    bar_type=BarType.from_str(BAR_TYPE_STR),
    sizing=SIZING,
    trade_notional=TRADE_NOTIONAL,
    ma_type=MA_TYPE,
    fast_period=FAST_MA,
    slow_period=SLOW_MA,
    liquidation=LIQ_RESOLVED,
)
strategy = MACross(config=config)

# Confirm the strategy received the resolved configs.
_s = strategy._sizing
print(f"Strategy._sizing      : mode={_s.mode}, fixed={_s.fixed_notional}, "
      f"risk_frac={_s.risk_frac}, stop_pct={_s.stop_pct}, "
      f"min_notional={_s.min_notional}")
print(f"Strategy._liq_config  : {strategy._liq_config}")

engine.add_strategy(strategy)
engine.run()
print()
print("Backtest complete.")

# Post-run state confirmation.
print(f"Post-run RiskEngine state    : {engine.kernel.risk_engine.trading_state}")
print(f"Position liquidations fired  : {len(position_liquidations)}")
print(f"Account liquidations fired   : {len(account_liquidations)}")
if position_liquidations:
    print()
    print("First 5 position liquidation events:")
    for ev in position_liquidations[:5]:
        print(f"  {ev}")
if account_liquidations:
    print()
    print("Account liquidation event:")
    for ev in account_liquidations:
        print(f"  {ev}")

In [ ]:
# ── Cell 5: Reports ──────────────────────────────────────────────—
fills_report     = engine.trader.generate_order_fills_report()
positions_report = engine.trader.generate_positions_report()
account_report   = engine.trader.generate_account_report(VENUE)

print(f"Order fills : {len(fills_report)}")
print(f"Positions   : {len(positions_report)}")

# Min balance + cross-check with liquidation events.
if not account_report.empty:
    totals = account_report["total"].astype(float)
    min_bal = totals.min()
    print(f"Min balance : {min_bal:.4f}")

    if LIQ_RESOLVED is not None and LIQ_RESOLVED.enabled:
        # With AccountAliveMonitor running, min_balance should not go far
        # below the configured floor. If it does, the actor missed a beat.
        if min_bal < -1.0 and not account_liquidations:
            print(f"⚠ min_balance={min_bal:.2f} but no AccountLiquidated event "
                  f"fired — actor missed an equity breach.")
        if account_liquidations and min_bal <= 0:
            print(f"  (min_bal ≤ 0 after AccountLiquidated is expected from "
                  f"residual fee debits.)")
    else:
        if min_bal <= 0:
            print(f"\n⚠ LIQUIDATED — min balance was {min_bal:.2f}")
            print("PnL results after liquidation are meaningless.")

print()
print("=== Recent Fills ===")
display(fills_report.tail(10))

print("\n=== Recent Positions ===")
display(positions_report.tail(10))

In [ ]:
# ── Cell 5b: Run diagnostics (orders + balance + margin) ──────────
#
# Standard "what just happened" panel.  Surfaces denied/rejected orders,
# the balance-curve high/low/final, lowest-N rows for drawdown anatomy,
# and a margin-field interpretation block.
print_run_diagnostics(
    engine, VENUE, instrument,
    trade_notional=TRADE_NOTIONAL,
    leverage=LEVERAGE,
)

In [ ]:
# ── Cell 6: Calculate statistics ──────────────────────────────────
#
# Must run before equity curve — analyzer.returns() is a getter that
# only has data after calculate_statistics() populates it.

account   = engine.cache.account_for_venue(VENUE)
positions = engine.cache.position_snapshots() + engine.cache.positions()
analyzer  = engine.portfolio.analyzer
analyzer.calculate_statistics(account, positions)
print(f"Stats calculated — {len(positions)} positions")

In [ ]:
# ── Cell 7: HTML Tearsheet (DISABLED — see notes) ────────────────
#
# NT's create_tearsheet() pulls from analyzer.get_performance_stats_returns()
# AND uses _calculate_daily_balance_returns() which applies the same
# upstream-broken zero-padding methodology (resample("D").last().ffill()
# .pct_change()).  About 80% of the tearsheet is therefore unreliable
# for sparse-trade strategies:
#
#   ❌ Returns Statistics block (Sharpe, Sortino, Vol, Returns PF,
#      Avg Return, Risk Return Ratio)
#   ❌ Equity Curve chart (built from cumulative broken returns)
#   ❌ Drawdown chart (derived from the same)
#   ❌ Monthly Returns heatmap (mostly zero-pad days)
#   ❌ Returns Distribution histogram
#   ❌ Rolling Sharpe Ratio (60d rolling on a mostly-zero series)
#   ❌ Yearly Returns chart (sums broken-series monthly returns)
#
# Only Run Information, Account Summary, and PnL Statistics are reliable
# — and all three are already shown elsewhere in this notebook with
# trustworthy methodology:
#
#   • PnL Statistics      → Cell 10  (print_summary_stats)
#   • Equity + drawdown   → Cell 9   (plot_equity_curve, event-time)
#   • Yearly breakdown    → Cell 10c (plot_yearly_breakdown)
#   • Trade distributions → Cell 10b (plot_trade_distributions)
#   • Regime breakdown    → Cell 10d (performance_by_regime)
#
# When upstream NT lands the daily-MTM fix (see
# docs/ANALYZER_RETURNS_CAVEAT.md), uncomment the create_tearsheet()
# block below to restore the integrated single-file report.
#
# from nautilus_trader.analysis import create_tearsheet
# html = create_tearsheet(
#     engine,
#     output_path=None,
#     title=(
#         f"{MA_TYPE}Cross({FAST_MA}/{SLOW_MA}) | {ASSET} {BAR_INTERVAL}"
#         f" | data: {DATA_SOURCE} | sim: {EXEC_VENUE}"
#     ),
# )
# display(HTML(html))
# if SAVE_TEARSHEET:
#     save_tearsheet(html, RESULT_NAME)

print("Cell 7 (NT tearsheet) is disabled — see comment above for why and where")
print("the trustworthy equivalents live in this notebook.")

In [ ]:
# ── Cell 8: Price chart with MA overlay + trade markers ──────────—

fig = plot_ma_cross(
    bars, fills_report,
    fast_period=FAST_MA,
    slow_period=SLOW_MA,
    ma_type=MA_TYPE,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
)
fig.show(config=dict(
    modeBarButtonsToRemove=["autoScale2d", "lasso2d", "select2d"],
    displaylogo=False,
))

In [ ]:
# ── Cell 9: Equity & drawdown (event-time) ────────────────────────
# NOTE: this is NOT a daily mark-to-market curve. NT's account report
# only records balance at event timestamps (fills, position changes).
# Between events, balance is held flat; intra-event price drift on
# open positions is invisible. Use this for drawdown anatomy and
# event-time PnL evolution, NOT for return-based statistics.
# See docs/ANALYZER_RETURNS_CAVEAT.md.
plot_equity_curve(
    account_report,
    f"{MA_TYPE}Cross({FAST_MA}/{SLOW_MA})  {INSTRUMENT_ID} {BAR_INTERVAL}",
    currency=CURRENCY,
)

In [ ]:
# ── Cell 10: Summary statistics ────────────────────────────────────
print_summary_stats(analyzer, num_positions=len(positions), currency=CURRENCY)

In [ ]:
# ── Cell 10b: Trade distributions ─────────────────────────────
#
# Three-panel trustworthy trade-quality view:
#  1. PnL distribution histogram (wins green, losers red, mean/median lines)
#  2. Trade duration histogram (bimodal = "two strategies in one")
#  3. Top-trade-share bars (% of total |PnL| from top/bottom trades —
#     concentration signals fragility).
plot_trade_distributions(
    positions,
    title=f"{MA_TYPE}Cross({FAST_MA}/{SLOW_MA})  {INSTRUMENT_ID} {BAR_INTERVAL}",
    bar_interval_ns=int(bars[1].ts_event - bars[0].ts_event) if len(bars) > 1 else None,
    currency=str(CURRENCY),
)

In [ ]:
# ── Cell 10c: Per-year breakdown ──────────────────────────────
#
# Year-over-year consistency.  Sign-flipping bars or diverging quality
# lines = regime-dependent strategy.
yearly = performance_by_year(positions, starting_capital=STARTING_CAPITAL)
print_yearly_breakdown(yearly, currency=str(CURRENCY))
plot_yearly_breakdown(
    yearly,
    title=f"Yearly performance — {MA_TYPE}Cross({FAST_MA}/{SLOW_MA})",
    currency=str(CURRENCY),
)

In [ ]:
# ── Cell 10d: Regime breakdown (ADX-tagged) ───────────────────────
#
# Tag every bar as TRENDING vs RANGING via ADX, then split realized PnL
# by the regime active at trade open.  Reveals what the strategy is
# actually being paid for.
regime_df = tag_regimes(bars, adx_period=14, adx_trending_threshold=25.0)
regime_perf = performance_by_regime(
    positions, regime_df, starting_capital=STARTING_CAPITAL,
)
if not regime_perf.empty:
    display(regime_perf.style.format({
        "pnl": "{:>10,.2f}", "pnl_pct": "{:>6.2f}%",
        "win_rate": "{:.2%}", "profit_factor": "{:>6.2f}",
        "avg_winner": "{:>8,.2f}", "avg_loser": "{:>8,.2f}",
        "avg_duration": "{:>5.1f}h",
    }))
else:
    print("No closed trades or no overlap between trades and regime data.")

In [ ]:
# ── Cell 10e: Comparison baselines ──────────────────────────────
#
# Two industry-standard "does my strategy actually have alpha?" checks:
#   1. Buy & hold (SPOT) — would just holding the asset have done better?
#   2. Random entry — does the strategy beat random with same trade count
#      and avg duration?  If not, no entry-timing alpha.
#
# baselines_for_strategy() handles position-introspection (extract trade
# count + average duration) and computes all three (spot B&H, leveraged
# B&H counterfactual, random-entry distribution).
baselines = baselines_for_strategy(
    positions, bars,
    starting_capital=float(STARTING_CAPITAL),
    notional_per_trade=float(TRADE_NOTIONAL),
    fee_rate=float(VENUE_CFG.taker_fee),
    leverage=float(LEVERAGE),
    n_simulations=1000,
    random_seed=42,
)

# Pre-compute the strategy PnL for the verdict + chart + tearsheet.
strategy_pnl = float(analyzer.total_pnl(CURRENCY))

print_baselines_verdict(
    baselines, strategy_pnl=strategy_pnl,
    leverage=LEVERAGE, currency=str(CURRENCY),
)

# Aliases for downstream cells (10f tearsheet, etc.)
bh_spot = baselines["buy_and_hold"]
random_dist = baselines["random_entry"]

plot_baselines_comparison(
    strategy_pnl=strategy_pnl,
    buy_and_hold_pnl=bh_spot["pnl"],
    random_entry_dist=random_dist,
    title=f"Strategy vs spot B&H + random — {MA_TYPE}Cross({FAST_MA}/{SLOW_MA})",
    currency=str(CURRENCY),
)

In [ ]:
# ── Cell 10f: v2 tearsheet (single-file shareable archive) ─────
#
# Self-contained HTML report that composes everything from cells 9–10e
# into one archivable file.  Uses ONLY trustworthy v2 metrics — the
# disabled cell 7 (NT tearsheet) is built on the upstream-broken returns
# methodology; this is the replacement.
#
# Saved to reports/tearsheets/{RESULT_NAME}_tearsheet_{timestamp}.html.
# Auto-opens in your default browser.
v2_tearsheet_path = generate_v2_tearsheet(
    positions=positions,
    account_report=account_report,
    bars=bars,
    starting_capital=STARTING_CAPITAL,
    currency=str(CURRENCY),
    instrument_label=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
    strategy_label=f"{MA_TYPE}Cross({FAST_MA}/{SLOW_MA})",
    leverage=LEVERAGE,
    fee_rate=float(VENUE_CFG.taker_fee),
    yearly_df=yearly,
    regime_df=regime_perf,
    baselines={
        "buy_and_hold": bh_spot,
        "random_entry": random_dist,
    },
    strategy_pnl=strategy_pnl,
    filename=f"{RESULT_NAME}_tearsheet",  # snapshot — appends _{ts}.html
    open_browser=True,
)
print(f"Saved: {v2_tearsheet_path}")

In [ ]:
# ── Cell 11b: Spotlight params (off-grid, mixed into the sweep) ──
#
# To test off-grid params (e.g. a Fibonacci pair) without breaking the
# heatmap's regular grid, tag them with `_kind: "spotlight"`.  The
# tag passes through to the sweep parquet but is stripped before
# strategy_factory sees the params.  Spotlight rows are visually
# badged in the sortable HTML report below; the heatmap silently
# excludes them via filter on `_kind != "spotlight"`.
#
# Example — uncomment to add off-grid combos to the sweep, then
# re-run Cell 11.
#
# spotlight_combos = [
#     {"fast": 8,  "slow": 21, "_kind": "spotlight", "_note": "8/21 EMA classic"},
#     {"fast": 9,  "slow": 18, "_kind": "spotlight", "_note": "9/18 trial"},
#     {"fast": 13, "slow": 21, "_kind": "spotlight", "_note": "Fibonacci"},
# ]
# all_combos = combos + spotlight_combos  # see Cell 11 sweep
print("Spotlight params guide — see comments in this cell.")

In [ ]:
# ── Cell 11: Parameter sweep ──────────────────────────────────────
#
# Grid search over fast/slow MA period combinations.
# Liquidation simulation runs per-engine when LIQ_RESOLVED is enabled.

def ma_factory(eng, params):
    cfg = MACrossConfig(
        instrument_id=instrument.id,
        bar_type=BarType.from_str(BAR_TYPE_STR),
        sizing=SIZING,
        trade_notional=TRADE_NOTIONAL,
        ma_type=MA_TYPE,
        fast_period=params["fast"],
        slow_period=params["slow"],
        liquidation=LIQ_RESOLVED,
    )
    eng.add_strategy(MACross(cfg))

combos = [{"fast": f, "slow": s} for f in FAST_PERIODS for s in SLOW_PERIODS if f < s]

results_df = run_sweep(
    venue=VENUE, instrument=instrument, bars=bars,
    starting_capital=STARTING_CAPITAL,
    param_combos=combos,
    strategy_factory=ma_factory,
    strategy_name=f"MACross-{MA_TYPE}-{EXEC_VENUE}",
    instrument_id=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
    sweep_name=SWEEP_NAME,
    leverage=LEVERAGE,
    liquidation=LIQ_RESOLVED,
    venue_config=VENUE_CFG,
    sizing=SIZING,
)

In [ ]:
# ── Cell 12: PnL heatmap ──────────────────────────────────────────
plot_pnl_heatmap(
    results_df, row_col="slow", col_col="fast",
    row_label=f"Slow {MA_TYPE} Period", col_label=f"Fast {MA_TYPE} Period",
    title=f"Total PnL ({CURRENCY}) by {MA_TYPE} Parameters",
)

In [ ]:
# ── Cell 12b: Sweep liquidation diagnostics ──────────────────────
#
# Cross-check the simulator's behaviour across the full sweep:
#  1. Schema completeness (every row has populated liq columns)
#  2. min_balance / liquidated_account consistency
#  3. Halt enforcement (dead combos should have post-halt denials)
#  4. Fee model cross-check (avg fees per position vs expected round-trip)
#  5. Liquidation-stop slippage (gap-risk surfaced)
print_sweep_liquidation_diagnostics(
    results_df,
    liq_resolved=LIQ_RESOLVED,
    trade_notional=TRADE_NOTIONAL,
)

In [ ]:
# ── Cell 12c: Sortable HTML sweep table ────────────────────────
#
# Self-contained interactive HTML report.  Click headers to sort,
# search any column, multi-column sort with shift-click, CSV export.
# Liquidated rows are highlighted red; spotlight rows are highlighted
# gold with a [SPOT] badge.
#
# Saved to reports/sweeps/{SWEEP_NAME}_sweep.html — deterministic
# filename mirrors the sweep parquet ({SWEEP_NAME}.parquet); re-running
# the same sweep grid overwrites both files.
#
# Auto-opens in your default browser (most reliable across Jupyter Lab,
# classic Jupyter, and VS Code/Cursor — VS Code's link handlers don't
# always cooperate with file:// URLs in iframes).
sweep_html_path = generate_sweep_html(
    results_df,
    filename=f"{SWEEP_NAME}_sweep",
    open_browser=True,
)

from IPython.display import HTML, display
display(HTML(
    f'<p style="font-size: 13px; padding: 12px; background: #f0f4ff; '
    f'border-left: 3px solid #2962ff; border-radius: 3px;">'
    f'📊 Sweep report opened in your default browser.<br>'
    f'<span style="color: #666; font-size: 11px;">Path: <code>{sweep_html_path}</code></span>'
    f'</p>'
))

In [ ]:
# -- Cell 13: TradingView Interactive Backtest ------------------
#
# Saved to reports/charts/{RESULT_NAME}_chart_{timestamp}.html — snapshot
# semantics, accumulates across runs of the same config.

path = generate_backtest_html(
    bars, fills_report, positions_report,
    fast_period=FAST_MA,
    slow_period=SLOW_MA,
    ma_type=MA_TYPE,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
    starting_capital=STARTING_CAPITAL,
    result_filename=f"{RESULT_NAME}_chart",
    open_browser=True,
)

In [ ]:
# ── Cell 13b: Robustness — alt-instrument sanity check ────────
#
# Run the chosen params on a few alternative instruments.  If the
# strategy is genuinely structural it should at least *not blow up*
# off-asset.  If it does, params are overfit to the primary asset.
#
# Disabled by default — uncomment and provide alt instruments to enable.
# Each entry needs: label, venue, instrument, bars, optional starting_capital.
#
# alt_instruments = []
# for asset in ["ETH", "SOL"]:
#     alt_id = make_instrument_id(asset, EXCHANGE)
#     alt_inst, alt_bars, _ = load_catalog_bars_for(alt_id, BAR_INTERVAL)
#     alt_instruments.append({
#         "label": asset, "venue": VENUE, "instrument": alt_inst,
#         "bars": alt_bars, "starting_capital": STARTING_CAPITAL,
#     })
# alt_df = run_alt_instrument_check(
#     instruments=alt_instruments,
#     params={"fast": FAST_MA, "slow": SLOW_MA, "ma_type": MA_TYPE},
#     strategy_factory=ma_factory,
#     leverage=LEVERAGE,
#     liquidation=LIQ_RESOLVED,
#     venue_config=VENUE_CONFIG,
#     sizing=SIZING,
# )
# display(alt_df[["label", "instrument_id", "total_pnl", "total_pnl_pct",
#                 "num_positions", "win_rate", "max_drawdown_pct",
#                 "pnl_profit_factor", "liquidated"]])
print("Alt-instrument check — see comments in this cell to enable.")

In [ ]:
# ── Cell 13c: Robustness — fee sensitivity ─────────────────────────
#
# Re-run chosen params at a range of fee levels (in bps).  Identifies
# the breakeven fee threshold and measures margin of safety.  A
# strategy that dies at 2x current fees has no slack for execution
# slippage in real markets.
from charts import plot_fee_sensitivity
fee_df = run_fee_sweep(
    venue=VENUE,
    instrument=instrument,
    bars=bars,
    starting_capital=STARTING_CAPITAL,
    params={"fast": FAST_MA, "slow": SLOW_MA, "ma_type": MA_TYPE},
    strategy_factory=ma_factory,
    leverage=LEVERAGE,
    verbose=False,
)
display(fee_df[["fee_bps", "total_pnl", "total_pnl_pct",
                "num_positions", "breakeven"]])
plot_fee_sensitivity(
    fee_df,
    title=f"Fee sensitivity — {MA_TYPE}Cross({FAST_MA}/{SLOW_MA})",
    currency=str(CURRENCY),
)

In [26]:
# ── Cell 14: Save notebook snapshot ────────────────────────────────────────
#
# Save the notebook (Ctrl+S) before running this cell.

#save_notebook("ema_cross.ipynb", RESULT_NAME)
#save_notebook_html("ema_cross.ipynb", RESULT_NAME)

In [ ]:
# ── Cell 15: Cleanup ──────────────────────────────────────────────
engine.dispose()
print("Engine disposed.")
